In [2]:
# ============================================================
# Supplementary Figure 5
# Gene composition of NMF-derived R-loop regulator programs
#
# Panel A: Complete row-scaled NMF loading heatmap
# Panel B: Top loading regulators for each RLP shown separately
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

# ============================================================
# 1. Path config
# ============================================================

PROJECT_DIR = "/mnt/zhangzheng_group/liuz-53/Test_R/R_loop_HCC"
FIG2_DIR = os.path.join(PROJECT_DIR, "02_Figure2_python")
SUPP_DIR = os.path.join(PROJECT_DIR, "02_Figure2_python")

os.makedirs(SUPP_DIR, exist_ok=True)

NMF_WEIGHT_FILE = os.path.join(
    FIG2_DIR,
    "Fig2_NMF_Rloop_program_gene_weights.csv"
)

OUT_PREFIX = os.path.join(
    SUPP_DIR,
    "SupplementaryFig5_RLP_gene_composition"
)

TOP_N = 15

# ============================================================
# 2. Plot setting
# ============================================================

mpl.rcParams["font.family"] = "DejaVu Sans"
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["axes.unicode_minus"] = False

# ============================================================
# 3. Load NMF gene-loading matrix
# ============================================================

H_df = pd.read_csv(NMF_WEIGHT_FILE, index_col=0)

# Expected input:
# rows = RLP1-RLP4
# columns = genes
loading_df = H_df.T.copy()

rlp_cols = [c for c in ["RLP1", "RLP2", "RLP3", "RLP4"] if c in loading_df.columns]
loading_df = loading_df[rlp_cols]

print("NMF loading matrix:")
print(loading_df.shape)

# ============================================================
# 4. Assign genes to RLPs by maximum loading
# ============================================================

gene_program = loading_df.idxmax(axis=1)
gene_max_loading = loading_df.max(axis=1)

gene_info = pd.DataFrame({
    "gene": loading_df.index,
    "assigned_program": gene_program.values,
    "max_loading": gene_max_loading.values
})

gene_info["program_order"] = gene_info["assigned_program"].map(
    {prog: i for i, prog in enumerate(rlp_cols)}
)

gene_info = gene_info.sort_values(
    ["program_order", "max_loading"],
    ascending=[True, False]
)

gene_info.to_csv(
    os.path.join(SUPP_DIR, "SupplementaryTable_S1_RLP_gene_loading_assignment.csv"),
    index=False
)

# ============================================================
# 5. Prepare heatmap matrix
# ============================================================

ordered_genes = gene_info["gene"].tolist()
heatmap_df = loading_df.loc[ordered_genes, :].copy()

# Row-scaled loading
row_mean = heatmap_df.mean(axis=1)
row_std = heatmap_df.std(axis=1).replace(0, np.nan)

heatmap_scaled = heatmap_df.sub(row_mean, axis=0).div(row_std, axis=0)
heatmap_scaled = heatmap_scaled.fillna(0)

# Boundaries between RLP-assigned gene blocks
program_assignments = gene_info["assigned_program"].values
boundaries = []

for i in range(1, len(program_assignments)):
    if program_assignments[i] != program_assignments[i - 1]:
        boundaries.append(i - 0.5)

# ============================================================
# 6. Prepare top loading genes for each RLP
# ============================================================

top_records = []

for prog in rlp_cols:
    top_genes = (
        loading_df[prog]
        .sort_values(ascending=False)
        .head(TOP_N)
    )

    for rank, (gene, weight) in enumerate(top_genes.items(), start=1):
        top_records.append({
            "program": prog,
            "rank": rank,
            "gene": gene,
            "weight": weight
        })

top_df = pd.DataFrame(top_records)

top_df.to_csv(
    os.path.join(SUPP_DIR, "SupplementaryFig5_top15_loading_genes_per_RLP.csv"),
    index=False
)

# ============================================================
# 7. Plot Supplementary Fig. 5
# ============================================================

program_colors = {
    "RLP1": "#D55E00",
    "RLP2": "#E69F00",
    "RLP3": "#0072B2",
    "RLP4": "#009E73"
}

fig = plt.figure(figsize=(12, 10.5))

outer_gs = fig.add_gridspec(
    nrows=2,
    ncols=1,
    height_ratios=[1.25, 1.75],
    hspace=0.38
)

# ----------------------------
# Panel A: complete heatmap
# ----------------------------

ax_heatmap = fig.add_subplot(outer_gs[0, 0])

im = ax_heatmap.imshow(
    heatmap_scaled.values,
    aspect="auto",
    cmap="RdBu_r",
    vmin=-2,
    vmax=2
)

ax_heatmap.set_xticks(np.arange(len(rlp_cols)))
ax_heatmap.set_xticklabels(rlp_cols, fontsize=12)

n_genes = heatmap_scaled.shape[0]

if n_genes <= 80:
    ax_heatmap.set_yticks(np.arange(n_genes))
    ax_heatmap.set_yticklabels(heatmap_scaled.index, fontsize=5)
else:
    step = max(1, n_genes // 45)
    yticks = np.arange(0, n_genes, step)
    ax_heatmap.set_yticks(yticks)
    ax_heatmap.set_yticklabels(heatmap_scaled.index[yticks], fontsize=5)

ax_heatmap.set_ylabel("R-loop regulators", fontsize=12)
ax_heatmap.set_title(
    "Complete gene-loading matrix of four NMF-derived R-loop regulator programs",
    fontsize=14,
    pad=10
)

for b in boundaries:
    ax_heatmap.axhline(b, color="black", linewidth=0.6)

cbar = fig.colorbar(
    im,
    ax=ax_heatmap,
    fraction=0.025,
    pad=0.02
)

cbar.set_label("Row-scaled NMF loading", fontsize=11)
cbar.ax.tick_params(labelsize=9)

ax_heatmap.text(
    -0.08,
    1.08,
    "A",
    transform=ax_heatmap.transAxes,
    fontsize=18,
    fontweight="bold",
    va="top"
)

# ----------------------------
# Panel B: 2 x 2 top genes barplots
# ----------------------------

inner_gs = outer_gs[1, 0].subgridspec(
    nrows=2,
    ncols=2,
    wspace=0.45,
    hspace=0.55
)

for idx, prog in enumerate(rlp_cols):

    ax = fig.add_subplot(inner_gs[idx // 2, idx % 2])

    tmp = (
        top_df[top_df["program"] == prog]
        .sort_values("weight", ascending=True)
        .copy()
    )

    y_pos = np.arange(tmp.shape[0])

    ax.barh(
        y_pos,
        tmp["weight"],
        color=program_colors.get(prog, "grey"),
        edgecolor="black",
        linewidth=0.3
    )

    ax.set_yticks(y_pos)
    ax.set_yticklabels(tmp["gene"], fontsize=8)

    ax.set_xlabel("NMF loading", fontsize=10)
    ax.set_title(
        f"{prog}: top {TOP_N} loading regulators",
        fontsize=11,
        fontweight="bold"
    )

    ax.tick_params(axis="x", labelsize=8)
    ax.tick_params(axis="y", labelsize=8)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.set_xlim(0, top_df["weight"].max() * 1.08)

# Panel B label
fig.text(
    0.045,
    0.56,
    "B",
    fontsize=18,
    fontweight="bold",
    va="top"
)

plt.tight_layout()

# ============================================================
# 8. Save outputs
# ============================================================

for ext in ["pdf", "png", "svg"]:
    fig.savefig(
        f"{OUT_PREFIX}.{ext}",
        dpi=300,
        bbox_inches="tight"
    )

plt.close(fig)

print("Done.")
print("Supplementary Fig. 5 saved to:")
print(SUPP_DIR)
print("Generated files:")
print(f"{OUT_PREFIX}.pdf")
print(f"{OUT_PREFIX}.png")
print(f"{OUT_PREFIX}.svg")

NMF loading matrix:
(1095, 4)


/tmp/ipykernel_2607928/2868791056.py:278: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


Done.
Supplementary Fig. 5 saved to:
/mnt/zhangzheng_group/liuz-53/Test_R/R_loop_HCC/02_Figure2_python
Generated files:
/mnt/zhangzheng_group/liuz-53/Test_R/R_loop_HCC/02_Figure2_python/SupplementaryFig5_RLP_gene_composition.pdf
/mnt/zhangzheng_group/liuz-53/Test_R/R_loop_HCC/02_Figure2_python/SupplementaryFig5_RLP_gene_composition.png
/mnt/zhangzheng_group/liuz-53/Test_R/R_loop_HCC/02_Figure2_python/SupplementaryFig5_RLP_gene_composition.svg
